# Similarity-aware Label Smoothing

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import CifarResNET18, CifarDenseNET121, CifarWideResNET2810
from metrics import top_label_ece, nll_loss
import random


## Hyperparams

In [2]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
dataset = "cifar100"
batch_size = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.1
epochs = 12
num_classes = int(dataset.removeprefix("cifar"))
epsilon = 0.05

## Training Utils

In [4]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [6]:
path = f"scores/{dataset}_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=5, alpha=10.0):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(torch.exp(-class_dists), dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    sims = sims ** alpha

    result = sims
    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [ ]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, num_classes=100, epochs=10, epsilon=0.1):
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

## RUN Experiments

In [8]:
# classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
# print(classwise_target)
# print(classwise_target.shape)

# classwise_target = torch.eye(n=num_classes, device=device)
# print(classwise_target)
# print(classwise_target.shape)
classwise_target = uniform_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target)
print(classwise_target.shape)

row_sums = torch.sum(classwise_target, dim=1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-10), "Rows do not sum to 1"

tensor([[9.5000e-01, 5.0505e-04, 5.0505e-04,  ..., 5.0505e-04, 5.0505e-04,
         5.0505e-04],
        [5.0505e-04, 9.5000e-01, 5.0505e-04,  ..., 5.0505e-04, 5.0505e-04,
         5.0505e-04],
        [5.0505e-04, 5.0505e-04, 9.5000e-01,  ..., 5.0505e-04, 5.0505e-04,
         5.0505e-04],
        ...,
        [5.0505e-04, 5.0505e-04, 5.0505e-04,  ..., 9.5000e-01, 5.0505e-04,
         5.0505e-04],
        [5.0505e-04, 5.0505e-04, 5.0505e-04,  ..., 5.0505e-04, 9.5000e-01,
         5.0505e-04],
        [5.0505e-04, 5.0505e-04, 5.0505e-04,  ..., 5.0505e-04, 5.0505e-04,
         9.5000e-01]], device='cuda:0')
torch.Size([100, 100])


In [9]:
train_loader, test_loader = get_data_loaders(dataset=dataset)
print(train_loader.dataset.classes)
model = CifarResNET18(num_classes=num_classes).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=200
)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    num_classes=num_classes,
    epochs=200,
    epsilon=epsilon,
)

100%|██████████| 169M/169M [00:05<00:00, 30.0MB/s] 


['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree',

Epoch 1/200 | Loss: 3.8547 | Test Acc: 0.1700 | Top-5 Test Acc: 0.4488


Epoch 2/200 | Loss: 3.1029 | Test Acc: 0.2679 | Top-5 Test Acc: 0.5775


Epoch 3/200 | Loss: 2.6020 | Test Acc: 0.3579 | Top-5 Test Acc: 0.6952


...

Epoch 196/200 | Loss: 0.4416 | Test Acc: 0.7993 | Top-5 Test Acc: 0.9444


Epoch 197/200 | Loss: 0.4418 | Test Acc: 0.7985 | Top-5 Test Acc: 0.9451


Epoch 198/200 | Loss: 0.4415 | Test Acc: 0.7998 | Top-5 Test Acc: 0.9453


Epoch 199/200 | Loss: 0.4415 | Test Acc: 0.7985 | Top-5 Test Acc: 0.9452


Epoch 200/200 | Loss: 0.4418 | Test Acc: 0.7978 | Top-5 Test Acc: 0.9453


In [10]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        logits_list.append(logits)
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)


In [11]:
print("ECE:", top_label_ece(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))
test_acc = accuracy(model, test_loader)
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

ECE: 0.08972946554422379
NLL: 0.9566543102264404
Top-1 Test Acc: 0.7978 | Top-5 Test Acc: 0.9453


In [12]:
PATH = f"ls_{seed}.pth"

torch.save(model.state_dict(), PATH)
